# 10 — Week 4: neuralforecast smoke test and timing (N-HiTS, PatchTST, DLinear, NLinear)

Purpose: before any full training run, check on this Mac that
1. the neuralforecast architectures run inside **our** protocol (`src/models/nf_models.py`: Week 3 loader, local scaling,
   masked loss on h = 1, 5, 15, early stopping on validation MAE),
2. training works end to end on **synthetic data** (no real data involved),
3. CPU vs MPS speed per model on the real modelling dataset (timing only, a few batches), and
4. one short real-data run (1 epoch on a subsample) gives sensible validation errors.

No test data is touched. Outputs: `reports/week4/nf_smoke.json`. Run the cells in order.

In [ ]:
# Cell 1: setup
import sys, json, time, platform, logging, traceback, warnings
from pathlib import Path
import numpy as np
import polars as pl
import torch

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
for name in ("pytorch_lightning", "lightning", "lightning.pytorch", "lightning_fabric"):
    logging.getLogger(name).setLevel(logging.ERROR)

import neuralforecast, pytorch_lightning
from models import rnn as R
from models import nf_models as NF

OUT = ROOT / "reports" / "week4"
OUT.mkdir(parents=True, exist_ok=True)
RES = {"versions": {"python": platform.python_version(), "torch": torch.__version__,
                    "neuralforecast": neuralforecast.__version__, "lightning": pytorch_lightning.__version__,
                    "numpy": np.__version__, "polars": pl.__version__},
       "mps_available": bool(torch.backends.mps.is_available()), "torch_threads": torch.get_num_threads()}
MODELS = ["nhits", "patchtst", "dlinear", "nlinear"]
CFG = {"seed": 0, "lr": 1e-3, "batch": 512, "epochs": 20, "patience": 3, "grad_clip": 1.0, "device": "cpu"}
DEVICES = ["cpu"] + (["mps"] if RES["mps_available"] else [])
print(json.dumps(RES, indent=1))
print("project root:", ROOT, "| models:", MODELS, "| devices:", DEVICES)


In [ ]:
# Cell 2: probe each architecture on a random batch (shape check, parameter count, CPU and MPS forward)
probe = []
for name in MODELS + ["gru"]:
    for dev in DEVICES:
        row = {"model": name, "device": dev}
        try:
            net, n_params = NF.build(name, CFG)
            net = net.to(torch.device(dev))
            x = torch.randn(8, R.INPUT_LEN, 1, device=torch.device(dev))
            out = net(x)
            loss = out.pow(2).mean()
            loss.backward()
            row.update(ok=True, n_params=n_params, out_shape=str(tuple(out.shape)),
                       layout=getattr(net, "layout", "rnn"), finite=bool(torch.isfinite(out).all()))
        except Exception as e:  # noqa: BLE001
            row.update(ok=False, error=f"{type(e).__name__}: {str(e)[:300]}")
            traceback.print_exc(limit=3)
        probe.append(row)
        print(row)
RES["probe"] = probe
print("\nALL PROBES OK" if all(r["ok"] for r in probe) else "\nSOME PROBES FAILED: send this output")


In [ ]:
# Cell 3: end-to-end training on SYNTHETIC data (no real data); every model should beat persistence clearly
def synth(n=30000, seed=0):
    """Minute series with mean reversion (AR(1) deviation), a slow level, a 5-minute dip, and invalid gaps."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    x = np.zeros(n)
    e = rng.normal(0, 0.4, n)
    for i in range(1, n):
        x[i] = 0.9 * x[i - 1] + e[i]
    P = 30 + np.cumsum(rng.normal(0, 0.01, n)) + x - 0.3 * (t % 5 == 0)
    valid = np.ones(n, bool)
    for g0 in rng.choice(np.arange(500, n - 500), 6, replace=False):
        valid[g0:g0 + 150] = False
    i_tr, i_va = int(0.7 * n), int(0.85 * n)
    split = np.where(t < i_tr, "train", np.where(t < i_va, "val", "test"))
    c = np.concatenate([[0], np.cumsum(~valid)])
    win_ok = np.zeros(n, bool)
    win_ok[R.INPUT_LEN - 1:] = (c[R.INPUT_LEN:] - c[:n - R.INPUT_LEN + 1]) == 0
    cols = {"minute_epoch": t * 60, "P_kW": np.where(valid, P, np.nan), "valid": valid, "split": split,
            "mod5": (t % 5).astype(np.int64), "hour_sin": np.sin(2 * np.pi * (t // 60 % 24) / 24),
            "hour_cos": np.cos(2 * np.pi * (t // 60 % 24) / 24)}
    for h in R.HORIZONS:
        ok = win_ok.copy()
        ok[n - h:] = False
        ok[:n - h] &= valid[h:]
        sp = np.full(n, None, dtype=object)
        tt = np.arange(n - h)
        sp[:n - h] = np.where(tt + h < i_tr, "train", np.where((tt >= i_tr) & (tt + h < i_va), "val",
                              np.where(tt >= i_va, "test", None)))
        ok &= sp != None  # noqa: E711
        cols[f"ok_h{h}"] = ok
        cols[f"split_h{h}"] = sp.tolist()
    df = pl.DataFrame(cols).with_columns(pl.col("P_kW").fill_nan(None))
    return df

sm = synth()
print("synthetic rows:", sm.height, "| valid share:", round(float(sm["valid"].mean()), 3),
      "| usable train/val origins h1:", int((sm["ok_h1"] & (sm["split_h1"] == "train")).sum()),
      int((sm["ok_h1"] & (sm["split_h1"] == "val")).sum()))
sdata = R.Data(sm, {"scaling": "local", "calendar": False, "covariates": False})
syn = []
for name in MODELS + ["gru"]:
    cfg = {**CFG, "model": name, "epochs": 3, "patience": 3, "max_train_samples": 8000}
    t0 = time.time()
    try:
        model, d, hist, info = NF.train(sm, cfg, log=lambda s: None, data=sdata)
        va = d.origins("val")
        pv = NF.predict(model, d, va, torch.device("cpu"))
        row = {"model": name, "ok": True, "n_params": info["n_params"], "s": round(time.time() - t0, 1)}
        for j, h in enumerate(R.HORIZONS):
            sel = np.isin(va, d.idx[h]["val"])
            y, p0 = d.P[va[sel] + h], d.P[va[sel]]
            m, mp = float(np.mean(np.abs(pv[sel, j] - y))), float(np.mean(np.abs(p0 - y)))
            row[f"skill_h{h}"] = round(1 - m / mp, 3)
        row["finite"] = bool(np.isfinite(pv).all())
    except Exception as e:  # noqa: BLE001
        row = {"model": name, "ok": False, "error": f"{type(e).__name__}: {str(e)[:300]}"}
        traceback.print_exc(limit=4)
    syn.append(row)
    print(row)
RES["synthetic"] = syn
good = all(r["ok"] and r["finite"] and r["skill_h1"] > 0.05 for r in syn)
print("\nSYNTHETIC CHECK PASSED (all models trained, finite, skill h1 > 0.05)" if good
      else "\nSYNTHETIC CHECK NOT PASSED: send this output")


In [ ]:
# Cell 4: timing on the REAL modelling dataset (a few batches per model and device; no full training)
md = pl.read_parquet(ROOT / "data" / "processed" / "model_dataset.parquet")
meta = json.loads((ROOT / "data" / "processed" / "model_dataset_meta.json").read_text())
data = R.Data(md, {"scaling": "local", "calendar": False, "covariates": False})
n_tr, n_va = len(data.origins("train")), len(data.origins("val"))
print(f"rows {md.height:,} | train origins {n_tr:,} | val origins {n_va:,} | scale s = {data.scale:.4f} kW")
timing = []
for name in MODELS + ["gru"]:
    for dev in DEVICES:
        try:
            r = NF.time_steps(name, CFG, data, torch.device(dev), n_batches=20)
            r = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()}
        except Exception as e:  # noqa: BLE001
            r = {"model": name, "device": dev, "error": f"{type(e).__name__}: {str(e)[:200]}"}
        timing.append(r)
        print(r)
RES["timing"] = timing
RES["n_train_origins"], RES["n_val_origins"] = n_tr, n_va
tdf = pl.DataFrame([t for t in timing if "error" not in t])
best = (tdf.sort("est_epoch_min").group_by("model", maintain_order=True).first()
          .select("model", "device", "n_params", "est_epoch_min"))
print("\nfastest device per model (estimated minutes per full epoch incl. validation):")
print(best)
RES["best_device"] = best.to_dicts()


In [ ]:
# Cell 5: one short REAL-data check per model: 1 epoch on 20,000 train origins, scored on 5,000 val origins
#          (sanity only: the numbers are NOT benchmark results)
dev_of = {r["model"]: r["device"] for r in RES["best_device"]}
real = []
for name in MODELS:
    cfg = {**CFG, "model": name, "epochs": 1, "patience": 1, "max_train_samples": 20000,
           "max_val_samples": 5000, "device": dev_of.get(name, "cpu")}
    t0 = time.time()
    try:
        model, d, hist, info = NF.train(md, cfg, log=print, data=data)
        rng = np.random.default_rng(1)
        va = np.sort(rng.choice(d.origins("val"), min(5000, len(d.origins("val"))), replace=False))
        pv = NF.predict(model, d, va, NF.pick_device(cfg["device"]))
        row = {"model": name, "device": cfg["device"], "s": round(time.time() - t0, 1)}
        for j, h in enumerate(R.HORIZONS):
            sel = np.isin(va, d.idx[h]["val"])
            y, p0 = d.P[va[sel] + h], d.P[va[sel]]
            row[f"MAE_h{h}"] = round(float(np.mean(np.abs(pv[sel, j] - y))), 4)
            row[f"pers_h{h}"] = round(float(np.mean(np.abs(p0 - y))), 4)
        row["finite"] = bool(np.isfinite(pv).all())
    except Exception as e:  # noqa: BLE001
        row = {"model": name, "error": f"{type(e).__name__}: {str(e)[:300]}"}
        traceback.print_exc(limit=4)
    real.append(row)
    print(row)
RES["real_short"] = real


In [ ]:
# Cell 6: save and summarise
(OUT / "nf_smoke.json").write_text(json.dumps(RES, indent=1, default=str))
print("written:", OUT / "nf_smoke.json")
for r in RES["best_device"]:
    est = r["est_epoch_min"]
    print(f"{r['model']:9s} {r['device']:4s} params {r['n_params']:>8,} | ~{est:5.1f} min/epoch | "
          f"~{est * 10 * 3 / 60:4.1f} h for 3 seeds x 10 epochs")
